# Amazon ML Challenge 2026 — all-in-one through Phase 2B

Attach only the raw competition dataset, enable Internet, select a CPU session, and choose **Run All**. This notebook clones one pinned repository commit and runs normalization → exact/structured retrieval → multikey blocking. Every stage is manifest-resumable within the same Kaggle working session.

Recommended: start a fresh Kaggle session. GPU is unnecessary. **UNVERIFIED total runtime estimate:** 45–150 minutes. Save a notebook version with outputs immediately after completion.

In [ ]:
from pathlib import Path
import json, subprocess, sys, time
REPO_URL='https://github.com/yugtheguy/NEW_AMAZON.git'
PINNED_COMMIT='2dc800d2e29c3583c9e15b427cd15743c8a29d7f'
INPUT_ROOT=Path('/kaggle/input')
WORKING_ROOT=Path('/kaggle/working')
REPO_DIR=WORKING_ROOT/'NEW_AMAZON'
CONFIG=REPO_DIR/'configs/prod.yaml'
print({'repo':REPO_URL,'commit':PINNED_COMMIT})

In [ ]:
expected={'train_source1.tsv','train_source2.tsv','train_source3.tsv','train_ground_truth.tsv','test_source1.tsv','test_source2.tsv','test_source3.tsv'}
found={name:list(INPUT_ROOT.rglob(name)) for name in sorted(expected)}
problems={name:paths for name,paths in found.items() if len(paths)!=1}
if problems: raise RuntimeError(f'Attach exactly one raw competition dataset. Problems: {problems}')
for name,paths in found.items(): print(name,paths[0],f'{paths[0].stat().st_size:,} bytes')

In [ ]:
if not REPO_DIR.exists(): subprocess.run(['git','clone','--filter=blob:none',REPO_URL,str(REPO_DIR)],check=True)
else:
    status=subprocess.check_output(['git','-C',str(REPO_DIR),'status','--porcelain'],text=True)
    if status.strip(): raise RuntimeError(f'Existing repository has tracked changes: {status}')
    subprocess.run(['git','-C',str(REPO_DIR),'fetch','origin'],check=True)
subprocess.run(['git','-C',str(REPO_DIR),'checkout','--detach',PINNED_COMMIT],check=True)
actual=subprocess.check_output(['git','-C',str(REPO_DIR),'rev-parse','HEAD'],text=True).strip()
expected_full=subprocess.check_output(['git','-C',str(REPO_DIR),'rev-parse',PINNED_COMMIT],text=True).strip()
assert actual==expected_full,(actual,expected_full)
subprocess.run([sys.executable,'-m','pip','install','-q','-e',f'{REPO_DIR}[data]','duckdb==1.5.5'],check=True)
verification=subprocess.check_output([sys.executable,'-c',"import duckdb,amazon_er.pipeline.multikey as m; print(duckdb.__version__); print(m.__file__)"],text=True)
print('Active commit:',actual); print(verification)
assert '1.5.5' in verification and str(REPO_DIR/'src/amazon_er/pipeline/multikey.py') in verification

In [ ]:
subprocess.run(['bash','-lc','free -h && df -h /kaggle/working'],check=True)
def run_stage(label,stage,extra=None):
    log_path=WORKING_ROOT/f'{label}.log'
    command=[sys.executable,'-m','amazon_er.cli','run','--stage',stage,'--config',str(CONFIG),*(extra or [])]
    print('\nRUNNING:', ' '.join(command),flush=True)
    started=time.monotonic()
    with log_path.open('a',encoding='utf-8') as log:
        process=subprocess.Popen(command,cwd=REPO_DIR,stdout=subprocess.PIPE,stderr=subprocess.STDOUT,text=True,bufsize=1)
        assert process.stdout is not None
        for line in process.stdout: print(line,end=''); log.write(line); log.flush()
        code=process.wait()
    if code: raise RuntimeError(f'{label} failed with exit code {code}; inspect {log_path}')
    print(f'{label} COMPLETE in {time.monotonic()-started:.1f}s',flush=True)
    subprocess.run(['bash','-lc','df -h /kaggle/working'],check=True)
    return log_path

In [ ]:
normalization_log=run_stage('phase1_normalization','normalize',['--data-root',str(INPUT_ROOT)])
NORMALIZED_ROOT=REPO_DIR/'artifacts/normalized/v1'
normalized_parts=list(NORMALIZED_ROOT.rglob('part-*.parquet')); normalized_manifests=list(NORMALIZED_ROOT.rglob('part-*.manifest.json'))
assert len(normalized_parts)==89,(len(normalized_parts),len(normalized_manifests)); assert len(normalized_manifests)==89
normalization_summary=json.loads((REPO_DIR/'artifacts/reports/normalization/v1/summary.json').read_text())
assert normalization_summary['cardinality_verified'] and normalization_summary['country_counts_verified']
print({'normalized_parquets':89,'normalization_manifests':89})

In [ ]:
phase2a_log=run_stage('phase2a_exact_structured','retrieve-exact-structured',['--normalized-root',str(NORMALIZED_ROOT),'--data-root',str(INPUT_ROOT)])
EXACT_ROOT=REPO_DIR/'artifacts/retrieval/v1/exact_structured'
exact_parts=list(EXACT_ROOT.rglob('part-*.parquet')); exact_manifests=list(EXACT_ROOT.rglob('part-*.manifest.json'))
assert len(exact_parts)==74,(len(exact_parts),len(exact_manifests)); assert len(exact_manifests)==74
exact_summary=json.loads((REPO_DIR/'artifacts/reports/retrieval/v1/exact_structured/summary.json').read_text())
assert exact_summary['status']=='complete' and exact_summary['train_recall'] is not None
print({'exact_structured_parquets':74,'union_recall':exact_summary['train_recall']['exact_union_structured_pair_recall']})

In [ ]:
phase2b_log=run_stage('phase2b_multikey','retrieve-multikey',['--normalized-root',str(NORMALIZED_ROOT),'--data-root',str(INPUT_ROOT)])
BLOCKER_ROOT=REPO_DIR/'artifacts/retrieval/v1/multikey_blocker'
blocker_parts=list(BLOCKER_ROOT.rglob('part-*.parquet')); blocker_manifests=list(BLOCKER_ROOT.rglob('part-*.manifest.json'))
assert len(blocker_parts)==74,(len(blocker_parts),len(blocker_manifests)); assert len(blocker_manifests)==74
blocker_summary=json.loads((REPO_DIR/'artifacts/reports/retrieval/v1/multikey_blocker/summary.json').read_text())
assert blocker_summary['status']=='complete' and blocker_summary['raw_key_coverage'] is not None
print(json.dumps({'phase1':{'parquets':89},'phase2a':{'parquets':74,'pair_recall':exact_summary['train_recall']['exact_union_structured_pair_recall']},'phase2b':{'parquets':74,'raw_key_coverage':blocker_summary['raw_key_coverage'],'pruned_pair_recall':blocker_summary['pruned_pair_recall'],'ranked_recall':blocker_summary['ranked_recall']},'logs':[str(normalization_log),str(phase2a_log),str(phase2b_log)]},indent=2))
subprocess.run(['bash','-lc','du -sh /kaggle/working/NEW_AMAZON/artifacts; df -h /kaggle/working'],check=True)
print('ALL STAGES THROUGH PHASE 2B COMPLETE — USE SAVE VERSION NOW')